# Fall-detection datasets: load + audit (WEDA-FALL, UMAFall, FallAllD)

**Goal:** load the three open datasets into pandas, show their shape and missing values, and judge how *relevant* and *clean* each one is for SilverGuard.

**What SilverGuard needs** (from `context.md`): a single **wrist-worn** MPU-6050 (accel + gyro) streaming at **50 Hz**, accel in **g**, gyro in **deg/s**, detecting falls in **older adults**. So "relevant" below means: wrist placement, accel + gyro, a usable sample rate, older-adult subjects, and fall classes that resemble what we will stage.

**How each dataset is represented**
- `*_trials` - one row per recording (metadata parsed from file names / headers + per-recording quality metrics).
- `df_*` - sample level (one row per sensor sample), keyed by `trial_id` which joins back to `*_trials`.

The notebook only **reads** the data. Nothing on disk is modified.

In [1]:
from pathlib import Path
import io, re, time, zipfile
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

DATA      = Path('../../data/open datasets')
WEDA_ROOT = DATA / 'WEDA-FALL' / 'dataset'
UMA_ROOT  = DATA / 'UMAFall' / 'UMAFall_Dataset' / 'UMAFall_Dataset'
UMA_FIXED_ZIP = DATA / 'UMAFall' / 'UMAFall_Dataset_corrected_version.zip'
FAD_ROOT  = DATA / 'FallAllD__zip' / 'FallAllD'
for p in (WEDA_ROOT, UMA_ROOT, UMA_FIXED_ZIP, FAD_ROOT):
    assert p.exists(), f'missing: {p.resolve()}'

TARGET_FS_HZ = 50     # SilverGuard sensor stream (context.md)

def missing_report(df):
    # per-column dtype, missing count and percentage
    miss = df.isna().sum()
    return pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': miss, 'missing_%': miss / max(len(df), 1) * 100})

def shape_line(df, name):
    mem = df.memory_usage(deep=True).sum() / 1e6
    print(f'{name}: shape = {df.shape[0]:,} rows x {df.shape[1]} cols  ({mem:,.0f} MB in memory)')

## 0. What is physically in each folder
Counts every file (recursively) so stray files, duplicates and redundant copies show up before we load anything.

In [2]:
def inventory(root, label):
    files = [p for p in root.rglob('*') if p.is_file()]
    ext = pd.Series([p.suffix.lower() or '(none)' for p in files]).value_counts().to_dict()
    stray = [p for p in files if '.ipynb_checkpoints' in p.parts]
    print(f'{label:10s} {len(files):>7,} files | by extension {ext} | stray .ipynb_checkpoints copies: {len(stray)}')
    return stray

weda_stray = inventory(WEDA_ROOT, 'WEDA-FALL')
uma_stray  = inventory(UMA_ROOT,  'UMAFall')
fad_stray  = inventory(FAD_ROOT,  'FallAllD')

print()
print('WEDA-FALL files per rate folder (the lower rates are re-sampled copies of 50Hz):')
print({d.name: sum(1 for _ in d.rglob('*.csv')) for d in sorted(WEDA_ROOT.iterdir()) if d.is_dir()})
print('UMAFall archives sitting next to the extracted folder:', sorted(p.name for p in UMA_FIXED_ZIP.parent.glob('*.zip')))
print('Non-data files shipped with FallAllD (metadata / readme):', [p.name for p in FAD_ROOT.iterdir() if p.suffix != '.dat'] or 'none')

WEDA-FALL   15,348 files | by extension {'.csv': 15348} | stray .ipynb_checkpoints copies: 0
UMAFall        748 files | by extension {'.csv': 748} | stray .ipynb_checkpoints copies: 2
FallAllD    26,420 files | by extension {'.dat': 26420} | stray .ipynb_checkpoints copies: 0

WEDA-FALL files per rate folder (the lower rates are re-sampled copies of 50Hz):
{'10Hz': 2907, '25Hz': 2907, '40Hz': 2907, '50Hz': 3719, '5Hz': 2907}
UMAFall archives sitting next to the extracted folder: ['UMAFall_Dataset.zip', 'UMAFall_Dataset_corrected_version.zip', 'videos_ADL.zip', 'videos_FALL.zip']
Non-data files shipped with FallAllD (metadata / readme): none


## 1. WEDA-FALL (wrist, Fitbit Sense, 50 Hz)
Layout: `dataset/<rate>/<activity>/U<user>_R<run>_<sensor>.csv`. We load the **50 Hz** folder (the others are down-sampled copies), sensors **accel** and **gyro** into `df_weda`. `orientation` and `vertical_accel` are only inventoried (an MPU-6050 does not give us those channels directly).

Note: `U??_R??_accel.csv` is matched exactly. A loose `*_accel.csv` would also match `*_vertical_accel.csv`.

In [3]:
def load_weda(root=WEDA_ROOT, rate='50Hz'):
    base = root / rate
    stamps = pd.read_csv(root / 'fall_timestamps.csv', encoding='utf-8-sig')   # file starts with a BOM
    stamps.columns = ['trial_key', 'fall_start_s', 'fall_end_s']

    trials, frames = [], []
    for act_dir in sorted(p for p in base.iterdir() if p.is_dir()):
        for acc_path in sorted(act_dir.glob('U??_R??_accel.csv')):
            stem = acc_path.name[:-len('_accel.csv')]
            tid = len(trials)
            rec = {'trial_id': tid, 'trial_key': f'{act_dir.name}/{stem}', 'activity': act_dir.name,
                   'user': int(stem[1:3]), 'run': int(stem[5:7])}
            for sensor in ('accel', 'gyro', 'orientation', 'vertical_accel'):
                p = act_dir / f'{stem}_{sensor}.csv'
                rec[f'has_{sensor}'] = p.exists()
                if not p.exists():
                    continue
                raw = pd.read_csv(p)
                rec[f'n_{sensor}'] = len(raw)
                rec[f'nan_{sensor}'] = int(raw.isna().sum().sum())
                if sensor in ('accel', 'gyro'):
                    raw.columns = ['t', 'x', 'y', 'z']
                    t = raw['t'].to_numpy()
                    dt = np.diff(t)
                    rec[f'dur_{sensor}_s'] = t.max() - t.min()
                    rec[f'backwards_t_{sensor}'] = int((dt < 0).sum())
                    rec[f'dup_t_{sensor}'] = int((dt == 0).sum())
                    if sensor == 'accel':
                        rec['fs_eff_hz'] = (len(t) - 1) / (t.max() - t.min())
                        rec['dt_p50_ms'], rec['dt_p90_ms'], rec['dt_max_ms'] = np.percentile(dt, [50, 90, 100]) * 1000
                        rec['clipped_frac'] = float((raw[['x', 'y', 'z']].abs().to_numpy() >= 39.1).mean())
                    frames.append(raw.assign(trial_id=tid, sensor=sensor))
            trials.append(rec)

    tr = pd.DataFrame(trials)
    tr['label'] = np.where(tr['activity'].str.startswith('F'), 'FALL', 'ADL')
    tr['group'] = np.where(tr['user'] >= 21, 'older', 'young')      # README: users 21-31 are 77-95 y
    tr = tr.merge(stamps, on='trial_key', how='left')
    df = pd.concat(frames, ignore_index=True)
    df = df.astype({'trial_id': 'int16', 'x': 'float32', 'y': 'float32', 'z': 'float32'})
    df['sensor'] = df['sensor'].astype('category')
    df = df[['trial_id', 'sensor', 't', 'x', 'y', 'z']]
    return tr, df, stamps

t0 = time.time()
weda_trials, df_weda, weda_stamps = load_weda()
print(f'loaded in {time.time()-t0:.0f}s')
shape_line(df_weda, 'df_weda')
print(f'weda_trials: {weda_trials.shape}')

loaded in 6s
df_weda: shape = 1,142,832 rows x 6 cols  (26 MB in memory)
weda_trials: (969, 32)


In [4]:
df_weda.head()

,trial_id,sensor,t,x,y,z
0,0,accel,0.0000,-0.8000,0.5000,9.9000
1,0,accel,0.0020,-0.7000,0.6000,9.8000
2,0,accel,0.0040,-0.6000,0.4000,9.9000
3,0,accel,0.0090,-0.5000,0.2000,10.2000
4,0,accel,0.1260,-0.7000,-0.8000,9.2000


In [5]:
print('Missing values in df_weda (sample level):')
display(missing_report(df_weda))
print('Rows per sensor:'); display(df_weda['sensor'].value_counts().to_frame('rows'))

Missing values in df_weda (sample level):


,dtype,missing,missing_%
trial_id,int16,0,0.0000
sensor,category,0,0.0000
t,float64,0,0.0000
x,float32,0,0.0000
y,float32,0,0.0000
z,float32,0,0.0000


Rows per sensor:


,rows
sensor,
accel,571460
gyro,571372


In [6]:
# Missing at the *file* level: which recordings lack a sensor, and NaNs inside the files that exist
has_cols = [c for c in weda_trials.columns if c.startswith('has_')]
nan_cols = [c for c in weda_trials.columns if c.startswith('nan_')]
print('Trials:', len(weda_trials))
print('Trials missing a sensor file:'); display((~weda_trials[has_cols]).sum().to_frame('missing_files'))
print('NaN cells inside the existing sensor files:'); display(weda_trials[nan_cols].fillna(0).sum().astype(int).to_frame('nan_cells'))
print('Trials missing vertical_accel, by activity:'); display(weda_trials.loc[~weda_trials.has_vertical_accel].groupby('activity').size().to_frame('trials').T)

Trials: 969
Trials missing a sensor file:


,missing_files
has_accel,0
has_gyro,0
has_orientation,0
has_vertical_accel,157


NaN cells inside the existing sensor files:


,nan_cells
nan_accel,0
nan_gyro,0
nan_orientation,0
nan_vertical_accel,0


Trials missing vertical_accel, by activity:


activity,D01,D03,D04,D09,D10,D11
trials,26,10,29,31,32,29


In [7]:
print('Class / population balance (trials):')
display(weda_trials.pivot_table(index='activity', columns='group', values='trial_id', aggfunc='count', fill_value=0, margins=True))
print('Fall trials per group:'); display(weda_trials.loc[weda_trials.label == 'FALL'].groupby('group').size().to_frame('fall_trials'))
print('Users:', weda_trials['user'].nunique(), '| young', weda_trials.loc[weda_trials.group=="young",'user'].nunique(), '| older', weda_trials.loc[weda_trials.group=="older",'user'].nunique())

Class / population balance (trials):


group,older,young,All
activity,,,
D01,26,42,68
D02,0,42,42
D03,10,42,52
D04,29,42,71
D05,0,42,42
D06,0,42,42
D07,0,42,42
D08,0,42,42
D09,31,42,73


Fall trials per group:


,fall_trials
group,
young,350


Users: 25 | young 14 | older 11


In [8]:
# Signal-level cleanliness
w = weda_trials
checks = {
    'accel rows != gyro rows (trials)':                int((w.n_accel != w.n_gyro).sum()),
    'trials with timestamps going backwards (accel)':  int((w.backwards_t_accel > 0).sum()),
    'trials with timestamps going backwards (gyro)':   int((w.backwards_t_gyro > 0).sum()),
    'trials with duplicate timestamps (accel)':        int((w.dup_t_accel > 0).sum()),
    'trials with accel clipped at +/-39.2 m/s2 (4 g)': int((w.clipped_frac > 0).sum()),
    '  ... of which fall trials':                      int(((w.clipped_frac > 0) & (w.label == 'FALL')).sum()),
    '  ... fall trials total':                         int((w.label == 'FALL').sum()),
}
display(pd.Series(checks, name='count').to_frame())
print('Effective sample rate per trial (Hz):'); display(w['fs_eff_hz'].describe().to_frame().T)
print('Gap between consecutive accel timestamps (ms), per-trial percentiles:'); display(w[['dt_p50_ms','dt_p90_ms','dt_max_ms']].describe().T)

,count
accel rows != gyro rows (trials),264
trials with timestamps going backwards (accel),24
trials with timestamps going backwards (gyro),32
trials with duplicate timestamps (accel),0
trials with accel clipped at +/-39.2 m/s2 (4 g),369
... of which fall trials,250
... fall trials total,350


Effective sample rate per trial (Hz):


,count,mean,std,min,25%,50%,75%,max
fs_eff_hz,969.0000,46.3847,4.6963,39.2055,39.8487,49.5739,49.7395,50.8931


Gap between consecutive accel timestamps (ms), per-trial percentiles:


,count,mean,std,min,25%,50%,75%,max
dt_p50_ms,969.0000,2.2059,0.4030,2.0000,2.0000,2.0000,2.0000,3.0000
dt_p90_ms,969.0000,90.8431,1.5142,87.6000,90.0000,90.0000,92.6000,94.0000
dt_max_ms,969.0000,141.2105,85.1093,102.0000,120.0000,124.0000,136.0000,"1,903.0000"


In [9]:
# Value ranges. Accel should be m/s2 (gravity ~9.8); gyro unit is not stated in the README.
acc = df_weda.loc[df_weda.sensor == 'accel', ['x', 'y', 'z']]
gyr = df_weda.loc[df_weda.sensor == 'gyro',  ['x', 'y', 'z']]
print('accel:'); display(acc.describe(percentiles=[.001, .5, .999]).T)
print('gyro:');  display(gyr.describe(percentiles=[.001, .5, .999]).T)
a = acc.to_numpy()
print('median |accel| (should be ~9.81 if m/s2):', round(float(np.median(np.linalg.norm(a, axis=1))), 2))
print('share of accel values that are exact multiples of 0.1 (i.e. 1-decimal resolution):', round(float(np.mean(np.abs(a * 10 - np.round(a * 10)) < 1e-3)), 4))

accel:


,count,mean,std,min,0.1%,50%,99.9%,max
x,"571,460.0000",-3.5590,6.1944,-39.0000,-37.8000,-3.9000,26.2000,39.2000
y,"571,460.0000",-2.9079,5.5530,-39.2000,-39.2000,-2.9000,20.9000,39.1000
z,"571,460.0000",3.3751,5.6548,-39.2000,-26.9000,3.3000,34.3000,39.2000


gyro:


,count,mean,std,min,0.1%,50%,99.9%,max
x,"571,372.0000",-0.0113,1.6306,-32.9000,-10.1000,-0.0000,10.3000,31.7000
y,"571,372.0000",0.0118,1.0880,-21.1000,-6.9000,-0.0000,6.8629,25.2000
z,"571,372.0000",-0.0008,1.2426,-25.6000,-7.1000,0.0000,7.1000,16.9000


median |accel| (should be ~9.81 if m/s2): 9.84
share of accel values that are exact multiples of 0.1 (i.e. 1-decimal resolution): 1.0


In [10]:
# Fall annotations (hand-labelled start/end of the fall; the README warns they can contain mistakes)
f = weda_trials.loc[weda_trials.label == 'FALL']
print('fall trials:', len(f), '| annotated:', int(f.fall_start_s.notna().sum()), '| annotation rows without a matching trial:', int((~weda_stamps.trial_key.isin(weda_trials.trial_key)).sum()))
dur = f.fall_end_s - f.fall_start_s
print('annotated fall length (s):'); display(dur.describe().to_frame().T)
print('annotation ends after the recording ends:', int((f.fall_end_s > f.dur_accel_s).sum()))

fall trials: 350 | annotated: 350 | annotation rows without a matching trial: 0
annotated fall length (s):


,count,mean,std,min,25%,50%,75%,max
0,350.0000,3.1037,0.2731,2.0000,3.0000,3.0000,3.0000,4.7000


annotation ends after the recording ends: 0


## 2. UMAFall (5 body positions incl. wrist, SensorTag ~20 Hz + phone ~200 Hz)
One CSV per trial with a `%`-prefixed metadata header, then `;`-separated samples:
`TimeStamp; Sample No; X; Y; Z; Sensor Type (0 accel, 1 gyro, 2 mag); Sensor ID`. Sensor ID -> body position is read from each file's own header.

The loader is deliberately tolerant: numbers that cannot be parsed become **NaN** (they are counted, not dropped), and only the top-level `*.csv` are read, so the stray `.ipynb_checkpoints` copies are ignored.

In [11]:
UMA_COLS = ['t_ms', 'sample_no', 'x', 'y', 'z', 'sensor_code', 'sensor_id']
UMA_SENSOR = {0: 'accel', 1: 'gyro', 2: 'mag'}

def parse_uma_text(text):
    lines = text.splitlines()
    k = next(i for i, l in enumerate(lines) if l.lstrip().startswith('% TimeStamp'))
    header, positions = {}, {}
    for l in lines[:k]:
        m = re.match(r'%\s*[0-9A-Fa-f:]{17};\s*(\d+);\s*(\w+);', l)      # "%MAC; sensor_id; POSITION; model"
        if m:
            positions[int(m.group(1))] = m.group(2)
            continue
        m = re.match(r'%\s*([^:;]+):\s*([^;]*?)\s*;*\s*$', l)             # "% Key: value" (+ optional ;;;; padding)
        if m:
            header.setdefault(m.group(1).strip(), m.group(2).strip())
    excel_padded = any(l.rstrip().endswith(';;;;;;;') for l in lines[:5])
    body = pd.read_csv(io.StringIO('\n'.join(lines[k + 1:])), sep=';', header=None, usecols=range(7), names=UMA_COLS)
    for c in UMA_COLS:
        body[c] = pd.to_numeric(body[c], errors='coerce')       # un-parseable -> NaN
    return header, positions, excel_padded, body

def uma_record(name, header, positions, excel_padded, body, tid):
    label_file = 'FALL' if '_fall_' in name.lower() else 'ADL'          # files are named ..._ADL_... / ..._Fall_...
    bad = body[['x', 'y', 'z']].isna().any(axis=1)
    m = re.search(r'Subject_(\d+)_(?:ADL|Fall)_(.+)_(\d+)_\d{4}-', name, flags=re.IGNORECASE)
    return {'trial_id': tid, 'file': name, 'subject': int(m.group(1)), 'activity': m.group(2), 'trial_no': int(m.group(3)),
            'label': label_file, 'label_header': header.get('Type of Movement'),
            'age': pd.to_numeric(header.get('Age'), errors='coerce'), 'gender': header.get('Gender'),
            'height_cm': pd.to_numeric(header.get('Height(cm)'), errors='coerce'), 'weight_kg': pd.to_numeric(header.get('Weight(Kg)'), errors='coerce'),
            'n_sensors_header': pd.to_numeric(header.get('Number of Sensors'), errors='coerce'),
            'positions_in_data': ','.join(sorted({positions.get(int(i), f'id{int(i)}') for i in body['sensor_id'].dropna().unique()})),
            'excel_padded_header': excel_padded, 'n_rows': len(body), 'n_unparseable_rows': int(bad.sum()),
            'duration_s': (body['t_ms'].max() - body['t_ms'].min()) / 1000}

def load_uma(root=UMA_ROOT):
    trials, frames = [], []
    for p in sorted(root.glob('*.csv')):                      # top level only -> skips .ipynb_checkpoints copies
        header, positions, excel_padded, body = parse_uma_text(p.read_text(encoding='utf-8', errors='replace'))
        tid = len(trials)
        trials.append(uma_record(p.name, header, positions, excel_padded, body, tid))
        body['trial_id'] = tid
        body['position'] = body['sensor_id'].map(lambda i: positions.get(int(i), f'id{int(i)}') if pd.notna(i) else None)
        frames.append(body)
    tr = pd.DataFrame(trials)
    df = pd.concat(frames, ignore_index=True)
    df['sensor'] = df['sensor_code'].map(UMA_SENSOR)
    df = df.astype({'trial_id': 'int16', 't_ms': 'int32', 'sample_no': 'int32', 'x': 'float32', 'y': 'float32', 'z': 'float32',
                    'sensor': 'category', 'position': 'category'})
    return tr, df[['trial_id', 'sensor', 'position', 't_ms', 'sample_no', 'x', 'y', 'z']]

t0 = time.time()
uma_trials, df_uma = load_uma()
print(f'loaded in {time.time()-t0:.0f}s')
shape_line(df_uma, 'df_uma')
print(f'uma_trials: {uma_trials.shape}')

loaded in 11s
df_uma: shape = 4,688,254 rows x 8 cols  (113 MB in memory)
uma_trials: (746, 17)


In [12]:
df_uma.head()

,trial_id,sensor,position,t_ms,sample_no,x,y,z
0,0,accel,RIGHTPOCKET,102,1,-0.1387,0.8869,0.3310
1,0,accel,RIGHTPOCKET,102,2,-0.1381,0.8865,0.3324
2,0,accel,RIGHTPOCKET,102,3,-0.1348,0.8896,0.3312
3,0,accel,RIGHTPOCKET,102,4,-0.1402,0.8866,0.3337
4,0,accel,RIGHTPOCKET,102,5,-0.1391,0.8863,0.3346


In [13]:
print('Missing values in df_uma (sample level):')
display(missing_report(df_uma))
bad_rows = df_uma[['x', 'y', 'z']].isna().any(axis=1)
print(f'rows with an un-parseable x/y/z: {int(bad_rows.sum()):,} of {len(df_uma):,} ({bad_rows.mean()*100:.2f}%)')
print('...by sensor:'); display(df_uma.loc[bad_rows].groupby('sensor', observed=True).size().to_frame('bad_rows'))
bad_trials = uma_trials.loc[uma_trials.n_unparseable_rows > 0]
print(f'trials affected: {len(bad_trials)} | subjects: {sorted(bad_trials.subject.unique())} | all header-padded by Excel: {bool(bad_trials.excel_padded_header.all())} | share of their rows lost: {bad_trials.n_unparseable_rows.sum()/bad_trials.n_rows.sum()*100:.0f}%')
display(bad_trials[['file', 'label', 'n_rows', 'n_unparseable_rows']].head(5))

Missing values in df_uma (sample level):


,dtype,missing,missing_%
trial_id,int16,0,0.0000
sensor,category,0,0.0000
position,category,0,0.0000
t_ms,int32,0,0.0000
sample_no,int32,0,0.0000
x,float32,27781,0.5926
y,float32,32750,0.6986
z,float32,25702,0.5482


rows with an un-parseable x/y/z: 47,067 of 4,688,254 (1.00%)
...by sensor:


,bad_rows
sensor,
accel,13384
gyro,16644
mag,17039


trials affected: 20 | subjects: [13] | all header-padded by Excel: True | share of their rows lost: 41%


,file,label,n_rows,n_unparseable_rows
392,UMAFall_Subject_13_ADL_Bending_1_2016-06-06_16...,ADL,5687,1920
393,UMAFall_Subject_13_ADL_Bending_1_2016-06-06_16...,ADL,5727,1744
394,UMAFall_Subject_13_ADL_Bending_2_2016-06-06_16...,ADL,5721,1700
395,UMAFall_Subject_13_ADL_Bending_3_2016-06-06_16...,ADL,5733,1801
396,UMAFall_Subject_13_ADL_GoDownstairs_1_2016-06-...,ADL,5730,2340


In [14]:
# Look at what the corruption is
sample = bad_trials.iloc[0]
raw_lines = (UMA_ROOT / sample.file).read_text(encoding='utf-8', errors='replace').splitlines()
k = next(i for i, l in enumerate(raw_lines) if l.startswith('% TimeStamp'))
bad_idx = next(i for i in range(k + 1, len(raw_lines)) if re.search(r'\d\.\d{3}\.\d{3}', raw_lines[i]))
print('Raw lines from a corrupted file (decimal points have been turned into thousands separators):')
print('\n'.join(l[:95] for l in raw_lines[bad_idx - 1:bad_idx + 3]))
print()
print('Is the corrected release fixed? Re-parse just those files from UMAFall_Dataset_corrected_version.zip (in memory, nothing extracted):')
with zipfile.ZipFile(UMA_FIXED_ZIP) as zf:
    members = {Path(n).name: n for n in zf.namelist() if n.endswith('.csv')}
    rows = []
    for name in bad_trials.file:
        _, _, padded, body = parse_uma_text(zf.read(members[name]).decode('utf-8', errors='replace'))
        rows.append({'file': name, 'excel_padded': padded, 'unparseable_rows': int(body[['x','y','z']].isna().any(axis=1).sum()),
                     'max_|accel|_g': body.loc[body.sensor_code == 0, ['x','y','z']].abs().to_numpy().max()})
fixed = pd.DataFrame(rows)
print(f'{len(fixed)} files re-read | unparseable rows now: {fixed.unparseable_rows.sum()} | worst accel magnitude: {fixed["max_|accel|_g"].max():.1f} g')

Raw lines from a corrupted file (decimal points have been turned into thousands separators):
194;13;0.03857284039258957;0.9916027784347534;0.3265874683856964;0;0;
194;14;0.06445082277059555;1.012.598.156.929.010;0.2596953213214874;0;0;
194;15;0.0952114462852478;1.010.156.750.679.010;0.202568456530571;0;0;
194;16;0.109859362244606;1.026.269.555.091.850;0.1693665236234665;0;0;

Is the corrected release fixed? Re-parse just those files from UMAFall_Dataset_corrected_version.zip (in memory, nothing extracted):
20 files re-read | unparseable rows now: 0 | worst accel magnitude: 7.2 g


In [15]:
print('Population / class balance:')
display(uma_trials.groupby('label').agg(trials=('trial_id', 'size'), subjects=('subject', 'nunique'), activities=('activity', 'nunique')))
display(uma_trials.pivot_table(index='activity', columns='label', values='trial_id', aggfunc='count', fill_value=0, margins=True))
subj = uma_trials.groupby('subject').agg(age=('age', 'first'), gender=('gender', 'first'), trials=('trial_id', 'size'),
                                         falls=('label', lambda s: int((s == 'FALL').sum())))
print('Subjects:'); display(subj.T)
print('Trials from the busiest subject:', f'{subj.trials.max()} of {subj.trials.sum()} ({subj.trials.max()/subj.trials.sum()*100:.0f}%)')

Population / class balance:


,trials,subjects,activities
label,,,
ADL,538,19,12
FALL,208,13,3


label,ADL,FALL,All
activity,,,
Aplausing,42,0,42
Bending,59,0,59
GoDownstairs,20,0,20
GoUpstairs,20,0,20
HandsUp,43,0,43
Hopping,53,0,53
Jogging,28,0,28
LyingDown_OnABed,57,0,57
MakingACall,46,0,46


Subjects:


subject,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
age,67,22,68,27,24,25,28,22,55,19,26,51,18,22,26,21,27,24,24
gender,F,F,M,M,M,M,M,F,M,M,M,F,F,F,M,M,F,M,F
trials,19,49,18,42,49,50,33,10,34,19,36,33,20,32,33,11,21,189,48
falls,0,19,0,12,18,17,6,6,0,0,18,0,0,9,12,6,11,56,18


Trials from the busiest subject: 189 of 746 (25%)


In [16]:
u = uma_trials
checks = {
    'stray checkpoint copies in the folder (excluded from load)': len(uma_stray),
    'trials whose header was Excel-padded / numbers mangled':   int(u.excel_padded_header.sum()),
    'trials with fewer than 5 body positions present in the data': int((u.positions_in_data.str.count(',') + 1 < 5).sum()),
    "trials where header 'Type of Movement' != file name":       int((u.label != u.label_header.str.strip(';').str.strip()).sum()),
    'duplicate (trial, sensor, position, sample_no) rows':      int(df_uma.duplicated(['trial_id', 'sensor', 'position', 'sample_no']).sum()),
}
display(pd.Series(checks, name='count').to_frame())
print('Trial duration (s):'); display(u.duration_s.describe().to_frame().T)

,count
stray checkpoint copies in the folder (excluded from load),2
trials whose header was Excel-padded / numbers mangled,20
trials with fewer than 5 body positions present in the data,233
trials where header 'Type of Movement' != file name,0
"duplicate (trial, sensor, position, sample_no) rows",0


Trial duration (s):


,count,mean,std,min,25%,50%,75%,max
duration_s,746.0000,14.8785,0.1029,14.1390,14.8532,14.8900,14.9187,15.0180


In [17]:
# Sample rate per sensor/position (samples / second, from the timestamps)
d = df_uma.dropna(subset=['position']).groupby(['trial_id', 'sensor', 'position'], observed=True).agg(n=('t_ms', 'size'), t0=('t_ms', 'min'), t1=('t_ms', 'max')).reset_index()
d['hz'] = d['n'] / ((d['t1'] - d['t0']) / 1000)
print('Effective rate (Hz) by sensor and body position:')
display(d.groupby(['sensor', 'position'], observed=True)['hz'].describe()[['count', 'mean', 'min', '50%', 'max']])
back = df_uma.groupby(['trial_id', 'sensor', 'position'], observed=True)['t_ms'].apply(lambda s: (s.diff() < 0).sum())
print('sensor streams with timestamps going backwards:', int((back > 0).sum()), 'of', len(back))

Effective rate (Hz) by sensor and body position:


count     mean      min      50%      max
sensor position                                                
accel  ANKLE       643.0000  20.0741  16.7199  20.1681  21.2963
       CHEST       745.0000  20.0761  16.7258  20.1654  21.2736
       RIGHTPOCKET 746.0000 201.7695 197.9460 200.0605 235.1127
       WAIST       617.0000  20.1336  16.7361  20.1667  21.1005
       WRIST       746.0000  20.0141   8.3813  20.1643  21.2917
gyro   ANKLE       643.0000  20.0763  16.7108  20.1686  21.2993
       CHEST       745.0000  20.0778  16.7247  20.1673  21.2751
       WAIST       617.0000  20.1350  16.7327  20.1673  21.1020
       WRIST       746.0000  20.0158   8.3819  20.1659  21.2933
mag    ANKLE       643.0000  20.0781  16.6848  20.1700  21.3024
       CHEST       745.0000  20.0796  16.7178  20.1681  21.2781
       WAIST       617.0000  20.1364  16.7293  20.1686  21.1020
       WRIST       746.0000  20.0176   8.3819  20.1667  21.2948

sensor streams with timestamps going backwards: 0 of 8999


In [18]:
# Value ranges per sensor, WRIST only (the position that matches SilverGuard)
wr = df_uma[df_uma.position == 'WRIST']
print('WRIST rows:', f'{len(wr):,}', '| trials with wrist data:', wr.trial_id.nunique(), 'of', len(uma_trials))
print(wr.groupby('sensor', observed=True)[['x', 'y', 'z']].describe(percentiles=[.001, .999]).T.to_string())
oor = df_uma[(df_uma.sensor == 'accel') & (df_uma[['x', 'y', 'z']].abs() > 16.5).any(axis=1)]
print(f'\naccel samples beyond the sensor +/-16 g range (physically impossible): {len(oor)} - all in trials from subjects {sorted(uma_trials.set_index("trial_id").loc[oor.trial_id.unique(), "subject"].unique())}')

WRIST rows: 662,964 | trials with wrist data: 746 of 746
sensor         accel         gyro          mag
x count 220,113.0000 216,204.0000 217,100.0000
  mean       -0.1680      -3.1794      36.8675
  std         0.8775      67.5060      37.8357
  min      -109.3750    -966.8750    -103.6667
  0.1%       -4.0476    -286.8750     -56.5000
  50%        -0.1689      -3.5859      38.3333
  99.9%       3.1702     253.9922     146.0000
  max         8.0000     996.8750     149.1667
y count 220,866.0000 216,970.0000 216,941.0000
  mean       -0.0201      -0.1148     -10.0430
  std         0.6120      55.0179      33.0706
  min        -8.0000    -996.8750     -91.0000
  0.1%       -4.1690    -256.0000     -64.0000
  50%         0.0195      -0.2969     -13.8333
  99.9%       3.3882     253.9922      63.8333
  max         7.9373     984.3750      92.3333
z count 220,906.0000 216,561.0000 217,202.0000
  mean        0.2850      -1.3440    -143.6587
  std         0.5515      59.9494      47.5544
  m

## 3. FallAllD (waist / wrist / neck loggers, 238 Hz accel+gyro)
File name = `S<subject>_D<device>_A<activity>_T<trial>_<sensor>.dat`, sensor `A` accel, `G` gyro, `M` magnetometer, `B` barometer. Files are header-less CSV of raw integer counts. No timestamps are stored.

`df_fad` holds **accel + gyro** (`ax ay az gx gy gz`, raw int16 counts) keyed by `trial_id`, `i` = sample index inside the trial. Barometer and magnetometer are read only to check them (row count / NaN) and are not kept.
Activity codes: `A001-A044` ADL, `A101-A135` falls (inferred from the code ranges present).

In [19]:
FAD_PAT = re.compile(r'^S(\d+)_D(\d)_A(\d+)_T(\d+)_([ABGM])\.dat$')

def read_dat(p):
    try:
        return pd.read_csv(p, header=None, dtype='int16').to_numpy()
    except pd.errors.EmptyDataError:
        return np.empty((0, 3), dtype='int16')
    except (ValueError, OverflowError):                     # NaN / non-integer / out of int16 -> fall back to float
        return pd.read_csv(p, header=None, dtype='float64').to_numpy()

def load_fad(root=FAD_ROOT):
    entries, unmatched = [], []
    for p in root.iterdir():
        m = FAD_PAT.match(p.name)
        if m:
            entries.append((int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4)), m.group(5), p))
        else:
            unmatched.append(p.name)
    idx = pd.DataFrame(entries, columns=['subject', 'device', 'activity', 'trial', 'sensor', 'path'])
    keys = ['subject', 'device', 'activity', 'trial']
    paths = idx.pivot(index=keys, columns='sensor', values='path').reset_index()

    trials, chunks, tids, sidx = [], [], [], []
    for tid, row in paths.iterrows():
        rec = {'trial_id': tid, **{k: int(row[k]) for k in keys}}
        arr = {}
        for s in 'ABGM':
            p = row.get(s)
            ok = isinstance(p, Path)
            rec[f'has_{s}'] = ok
            a = read_dat(p) if ok else np.empty((0, 3), dtype='int16')
            rec[f'n_{s}'] = len(a)
            rec[f'nan_{s}'] = int(np.isnan(a).sum()) if a.dtype.kind == 'f' else 0
            arr[s] = a
        A, G = arr['A'], arr['G']
        rec['accel_at_rail'] = bool(len(A) and np.abs(A).max() >= 32760)      # +/-8 g full scale reached
        rec['gyro_at_rail']  = bool(len(G) and np.abs(G).max() >= 32760)
        rec['rest_norm_counts'] = float(np.median(np.linalg.norm(A.astype('float32'), axis=1))) if len(A) else np.nan
        n = min(len(A), len(G))
        if n:
            chunks.append(np.hstack([A[:n], G[:n]]))
            tids.append(np.full(n, tid, dtype='int16'))
            sidx.append(np.arange(n, dtype='int16'))
        trials.append(rec)

    tr = pd.DataFrame(trials)
    tr['label'] = np.where(tr['activity'] >= 100, 'FALL', 'ADL')
    big = np.concatenate(chunks)
    df = pd.DataFrame(big, columns=['ax', 'ay', 'az', 'gx', 'gy', 'gz'])
    df.insert(0, 'i', np.concatenate(sidx))
    df.insert(0, 'trial_id', np.concatenate(tids))
    return tr, df, unmatched

t0 = time.time()
fad_trials, df_fad, fad_unmatched = load_fad()
print(f'loaded in {time.time()-t0:.0f}s | files not matching the naming pattern: {len(fad_unmatched)}')
shape_line(df_fad, 'df_fad')
print(f'fad_trials: {fad_trials.shape}')

loaded in 36s | files not matching the naming pattern: 0
df_fad: shape = 31,439,800 rows x 8 cols  (503 MB in memory)
fad_trials: (6605, 21)


In [20]:
df_fad.head()

,trial_id,i,ax,ay,az,gx,gy,gz
0,0,0,4155,55,-82,1613,205,77
1,0,1,4157,55,-126,1626,202,74
2,0,2,4157,50,-175,1634,199,70
3,0,3,4152,46,-221,1632,207,64
4,0,4,4150,42,-262,1623,213,61


In [21]:
print('Missing values in df_fad (sample level):')
display(missing_report(df_fad))
nan_cols = [c for c in fad_trials.columns if c.startswith('nan_')]
print('NaN cells in the files that were read (A, B, G, M):'); display(fad_trials[nan_cols].sum().to_frame('nan_cells').T)
has_cols = [c for c in fad_trials.columns if c.startswith('has_')]
print('Trials missing a sensor file:'); display((~fad_trials[has_cols]).sum().to_frame('missing_files').T)
n_cols = [c for c in fad_trials.columns if c.startswith('n_') and len(c) == 3]
print('Rows per file, by sensor (min / max):'); display(fad_trials[n_cols].agg(['min', 'max']))

Missing values in df_fad (sample level):


,dtype,missing,missing_%
trial_id,int16,0,0.0000
i,int16,0,0.0000
ax,int16,0,0.0000
ay,int16,0,0.0000
az,int16,0,0.0000
gx,int16,0,0.0000
gy,int16,0,0.0000
gz,int16,0,0.0000


NaN cells in the files that were read (A, B, G, M):


,nan_A,nan_B,nan_G,nan_M
nan_cells,0,0,0,0


Trials missing a sensor file:


,has_A,has_B,has_G,has_M
missing_files,0,0,0,0


Rows per file, by sensor (min / max):


,n_A,n_B,n_G,n_M
min,4760,200,4760,1600
max,4760,200,4760,1600


In [22]:
print('Population / class balance (trials, each trial = 1 device recording):')
display(fad_trials.groupby('label').agg(trials=('trial_id', 'size'), subjects=('subject', 'nunique'), activities=('activity', 'nunique')))
display(fad_trials.pivot_table(index='device', columns='label', values='trial_id', aggfunc='count', margins=True))
print('trials per subject:'); display(fad_trials.groupby('subject').size().to_frame('trials').T)
g = (fad_trials.groupby(['subject', 'device', 'activity']).trial.agg(['min', 'max', 'count']))
print('(subject, device, activity) groups whose trial numbers have gaps:', int(((g['max'] - g['min'] + 1) != g['count']).sum()), 'of', len(g))

Population / class balance (trials, each trial = 1 device recording):


,trials,subjects,activities
label,,,
ADL,4883,15,44
FALL,1722,13,35


label,ADL,FALL,All
device,,,
1,1559,733,2292
2,1992,523,2515
3,1332,466,1798
All,4883,1722,6605


trials per subject:


subject,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
trials,531,593,578,302,543,234,367,285,511,334,417,524,494,524,368


(subject, device, activity) groups whose trial numbers have gaps: 108 of 2277


In [23]:
# Scale / saturation. At rest |accel| should be ~1 g; its value in counts gives the LSB/g
rest = fad_trials.loc[fad_trials.label == 'ADL', 'rest_norm_counts']
print(f'median |accel| over ADL trials = {rest.median():,.0f} counts  ->  ~{rest.median():,.0f} counts/g  (4096 counts/g = +/-8 g full scale)')
print('trials whose accel hit the +/-8 g rail (sum = trials at rail, size = trials):'); display(fad_trials.groupby(['label', 'device']).accel_at_rail.agg(['sum', 'size']).T)
print('trials whose gyro hit the rail:', int(fad_trials.gyro_at_rail.sum()))
print('accel / gyro raw ranges:'); display(df_fad[['ax','ay','az','gx','gy','gz']].agg(['min', 'max']))

median |accel| over ADL trials = 4,061 counts  ->  ~4,061 counts/g  (4096 counts/g = +/-8 g full scale)
trials whose accel hit the +/-8 g rail (sum = trials at rail, size = trials):


label    ADL             FALL          
device     1     2     3    1    2    3
sum        0     0     0    0    0   17
size    1559  1992  1332  733  523  466

trials whose gyro hit the rail: 1
accel / gyro raw ranges:


,ax,ay,az,gx,gy,gz
min,-32764,-32764,-32764,-30568,-32768,-27849
max,32764,32764,32764,32715,32699,24621


In [24]:
# Where inside the fixed 20 s window does the biggest accel magnitude sit? (238 Hz assumed: 4760 samples = 20 s)
FAD_FS = 238
n_per = int(fad_trials.n_A.iloc[0]); assert (fad_trials.n_A == n_per).all() and (fad_trials.n_G == n_per).all()
acc = df_fad[['ax', 'ay', 'az']].to_numpy(dtype=np.float32).reshape(len(fad_trials), n_per, 3)
fad_trials['peak_t_s'] = np.linalg.norm(acc, axis=2).argmax(axis=1) / FAD_FS
del acc
print(f'window = {n_per/FAD_FS:.1f} s | time of peak accel magnitude (s):')
display(fad_trials.groupby('label').peak_t_s.describe(percentiles=[.1, .5, .9]))

window = 20.0 s | time of peak accel magnitude (s):


,count,mean,std,min,10%,50%,90%,max
label,,,,,,,,
ADL,"4,883.0000",10.1127,4.6983,0.0000,2.5714,10.0042,16.8109,19.9958
FALL,"1,722.0000",9.9041,1.2579,0.0252,9.8239,10.0042,10.0336,19.7143


## 4. Side-by-side summary
Everything here is computed from the DataFrames above, except sensor placement / unit rows that come from the dataset READMEs or the paper and cannot be checked from the files.

In [25]:
def n_missing(df): return int(df.isna().sum().sum())

older_uma = uma_trials[uma_trials.age >= 60]
rows = {
    'WEDA-FALL': {
        'sample rows (df)': len(df_weda), 'missing cells (df)': n_missing(df_weda),
        'trials': len(weda_trials), 'subjects': weda_trials.user.nunique(),
        'fall trials': int((weda_trials.label == 'FALL').sum()), 'ADL trials': int((weda_trials.label == 'ADL').sum()),
        'fall types / ADL types': f"{weda_trials.loc[weda_trials.label=='FALL','activity'].nunique()} / {weda_trials.loc[weda_trials.label=='ADL','activity'].nunique()}",
        'older subjects (>=60 y)': int(weda_trials.loc[weda_trials.group == 'older', 'user'].nunique()),
        'falls recorded by older subjects': int(((weda_trials.group == 'older') & (weda_trials.label == 'FALL')).sum()),
        'sensor placement': 'wrist', 'sample rate (Hz)': f"~{weda_trials.fs_eff_hz.median():.0f} (irregular)",
        'accel unit': 'm/s2 (clipped at 4 g)', 'gyro unit': 'not documented', 'has gyro': True,
    },
    'UMAFall': {
        'sample rows (df)': len(df_uma), 'missing cells (df)': n_missing(df_uma),
        'trials': len(uma_trials), 'subjects': uma_trials.subject.nunique(),
        'fall trials': int((uma_trials.label == 'FALL').sum()), 'ADL trials': int((uma_trials.label == 'ADL').sum()),
        'fall types / ADL types': f"{uma_trials.loc[uma_trials.label=='FALL','activity'].nunique()} / {uma_trials.loc[uma_trials.label=='ADL','activity'].nunique()}",
        'older subjects (>=60 y)': int(older_uma.subject.nunique()),
        'falls recorded by older subjects': int((older_uma.label == 'FALL').sum()),
        'sensor placement': 'wrist + 4 other positions', 'sample rate (Hz)': '~20 (SensorTags), ~200 (phone)',
        'accel unit': 'g', 'gyro unit': 'not stated in file', 'has gyro': True,
    },
    'FallAllD': {
        'sample rows (df)': len(df_fad), 'missing cells (df)': n_missing(df_fad),
        'trials': len(fad_trials), 'subjects': fad_trials.subject.nunique(),
        'fall trials': int((fad_trials.label == 'FALL').sum()), 'ADL trials': int((fad_trials.label == 'ADL').sum()),
        'fall types / ADL types': f"{fad_trials.loc[fad_trials.label=='FALL','activity'].nunique()} / {fad_trials.loc[fad_trials.label=='ADL','activity'].nunique()}",
        'older subjects (>=60 y)': 'unknown (no metadata file)',
        'falls recorded by older subjects': 'unknown (no metadata file)',
        'sensor placement': 'waist/wrist/neck (mapping not in folder)', 'sample rate (Hz)': '238 (fixed, no timestamps)',
        'accel unit': 'raw counts (~4096/g)', 'gyro unit': 'raw counts', 'has gyro': True,
    },
}
summary = pd.DataFrame(rows)
display(summary)

,WEDA-FALL,UMAFall,FallAllD
sample rows (df),1142832,4688254,31439800
missing cells (df),0,86233,0
trials,969,746,6605
subjects,25,19,15
fall trials,350,208,1722
ADL trials,619,538,4883
fall types / ADL types,8 / 11,3 / 12,35 / 44
older subjects (>=60 y),11,2,unknown (no metadata file)
falls recorded by older subjects,0,0,unknown (no metadata file)
sensor placement,wrist,wrist + 4 other positions,waist/wrist/neck (mapping not in folder)


## 5. Evaluation: relevance and cleanliness

Ratings are against SilverGuard's target (wrist MPU-6050, accel + gyro, 50 Hz, older adults, high fall recall). Numbers quoted are the ones printed above.

| | **WEDA-FALL** | **UMAFall** | **FallAllD** |
|---|---|---|---|
| **Relevance** | **High** - only native wrist + ~50 Hz set | **Medium-low** - wrist is only ~20 Hz | **Medium-high** - biggest and richest, wrist subset needs confirming |
| **Cleanliness** | **Medium** - no NaN, but messy timing and clipping | **Low as shipped**, good with the corrected zip | **High** - essentially perfect structure |
| **Size (df rows)** | 1.14 M | 4.69 M | 31.4 M |
| **Falls / ADL trials** | 350 / 619 | 208 / 538 | 1,722 / 4,883 |

### WEDA-FALL
**Relevance - good fit for the hardware, weak on who fell.**
- Matches the target best: wrist worn, accel + gyro, ~50 Hz, and 8 fall types (slips, trips, falls from sitting/fainting) that resemble real elderly falls.
- It has older adults (11 users, 77-95 y per the README) but they recorded **ADLs only (157 trials) and no falls**. All 350 falls come from 14 younger people (one is 46) onto a mattress. So older-adult data is useful for *false-alarm* testing, not for fall recall.
- **Accel is clipped at +/-39.2 m/s2 (4 g)**: 369 of 969 trials touch the limit, including **250 of 350 fall trials (71%)**. The impact peak is exactly the feature a threshold/ML fall rule relies on, and our MPU-6050 will probably have a wider range, so models trained here will see a distribution shift.
- Accel is in m/s2 (median |a| = 9.84), resolution is only 0.1 m/s2. The README does not give the gyro unit; its magnitudes (99.9% within +/-10, max 33) look like **rad/s**, not deg/s - confirm before converting.

**Cleanliness - no missing values, but the time axis cannot be trusted as-is.**
- 0 NaN; 350/350 falls have start/end annotations (hand-labelled, README warns they may contain errors); labels consistent.
- Samples arrive in **bursts**: median gap 2 ms but 90th percentile ~90 ms, so the "50 Hz" stream is really ~10 batches per second. Effective rate per trial is 39-51 Hz (two groups, ~40 and ~50).
- Accel and gyro are **not sample-aligned**: 264 of 969 trials have different row counts, and 24 (accel) / 32 (gyro) trials have timestamps that go backwards.
- Needs: sort by time, resample each stream onto a uniform 50 Hz grid, then align accel and gyro, before any windowing.

### UMAFall
**Relevance - useful as a secondary set, mostly not at the right rate.**
- Every trial has a **WRIST** accel + gyro + mag stream, plus chest, waist, ankle and a pocket phone. Accel is in g (same as our contract). Good variety of hard negatives (hopping, jogging, lying down, sitting) and 3 fall types (forward, backward, lateral).
- But the wrist SensorTag runs at **~20 Hz**, well under the 50 Hz target; up-sampling cannot recover the impact shape. Only the pocket phone (accel only) is fast (~200 Hz).
- 19 subjects, ages 18-68; **2 subjects are >= 60 and neither has a fall** (the 51- and 55-year-olds also have none). One subject (18) provides 25% of all trials and 56 of the 208 falls, so a random split would leak badly - split by subject.
- All trials are a fixed ~15 s with no event annotation, so impact time has to be found from the signal.

**Cleanliness - needs the corrected release.**
- **20 of 746 trials (all Subject 13, all ADL) were damaged by an Excel round-trip**: decimal points became thousands separators (e.g. `1.012.598.156.929.010`). 47,067 rows (1.0% of the dataset, **41% of those 20 files**) cannot be parsed, and 24 accel samples that *do* parse sit far outside the sensor's +/-16 g range. `UMAFall_Dataset_corrected_version.zip` (next to the folder) fixes exactly these 20 files: re-reading them gives 0 unparseable rows. Use the corrected release.
- 2 stray `.ipynb_checkpoints` duplicates sit inside the folder (748 vs the real 746 CSVs) and would double-count if globbed recursively.
- Otherwise clean: no duplicate rows, timestamps always increase, header label always agrees with the file name. 233 trials lack one or more of the 5 body positions (waist/ankle most often) - irrelevant for wrist work.

### FallAllD
**Relevance - the strongest raw material, a few things to confirm.**
- Largest and most varied: 15 subjects, **35 fall types and 44 ADL types**, 1,722 fall trials, accel + gyro at **238 Hz**, so it can be decimated to 50 Hz (with an anti-alias filter) and even used to mimic the MPU-6050.
- Three devices per recording; the wrist logger (believed to be `D2`, 523 falls / 1,992 ADL trials) is the subset that matches us. **The device -> body-position mapping, activity names, subject ages and gyro scale are not in the folder** (no metadata file shipped) - take them from the dataset paper. The paper reports adult volunteers (I recall ages ~21-53; unverified here), so assume **no older-adult falls**.
- Accel is raw counts: resting |a| = 4,061 counts, i.e. ~4,096 counts/g (+/-8 g range) - consistent with the data. Gyro scale is unverified.
- **Event-position bias:** in fall trials the biggest accel peak sits at **~10.0 s of the 20 s window (10th-90th percentile: 9.8-10.0 s)**, whereas ADL peaks are spread across the whole window. The recordings were trigger-centred. A model that sees whole trials can learn "peak in the middle = fall" instead of real fall dynamics. Cut windows from the whole 20 s and label by distance to the impact.

**Cleanliness - best of the three.**
- 26,420 files all match the naming pattern; every one of the 6,605 trials has all four sensors; every file has exactly the expected length (4,760 / 200 / 4,760 / 1,600 rows); 0 NaN; no stray files.
- Minor: **17 fall trials from device 3 hit the +/-8 g rail** (clipped), 1 gyro trial hit its rail, 108 of 2,277 (subject, device, activity) groups have gaps in trial numbers, and subjects are unbalanced (234 to 593 trials). There are no timestamps, so a fixed 238 Hz is assumed (4,760 samples = 20 s exactly).

### What this means for the model
1. **Core training/benchmark:** FallAllD wrist subset, decimated to 50 Hz, converted to g and deg/s. **Cross-check on WEDA-FALL** (native wrist 50 Hz, older-adult ADLs for false-alarm testing). Treat UMAFall as a third, lower-priority check, from the corrected zip.
2. **No dataset contains an older adult falling.** This matches the project plan (staged falls by healthy teammates on a mattress) but belongs in the report as a limitation.
3. **Harmonise before combining:** one unit set (g, deg/s), one rate (50 Hz, uniform grid), flag or drop clipped trials (WEDA 4 g, FallAllD 17 trials), check the axis conventions of each device before merging.
4. **Split by subject, never by window** (context.md), and do not let the FallAllD "peak at 10 s" pattern into the features.
5. **Still to verify from outside the folder:** FallAllD device-to-position mapping and gyro scale; the WEDA gyro unit.